In [2]:
import os

import numpy as np
import pandas as pd
import yfinance as yf

# ------------------------------------------------------------------------------
# 1. DEFINE TARGET TICKER UNIVERSE WITH METADATA TAGS
# ------------------------------------------------------------------------------
# Mapping tickers to Sector/Category for downstream comparative analysis
TICKER_METADATA = {
    # --- 2008 Global Financial Crisis (Distressed & Anchors) ---
    "LEHMQ": {
        "Name": "Lehman Brothers",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "BSC": {
        "Name": "Bear Stearns",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "AIG": {
        "Name": "American International Group",
        "Category": "Distressed_2008",
        "Sector": "Financials",
    },
    "C": {"Name": "Citigroup", "Category": "Distressed_2008", "Sector": "Financials"},
    "JPM": {
        "Name": "JPMorgan Chase",
        "Category": "Anchor_Financial",
        "Sector": "Financials",
    },
    "BAC": {
        "Name": "Bank of America",
        "Category": "Anchor_Financial",
        "Sector": "Financials",
    },
    "GS": {
        "Name": "Goldman Sachs",
        "Category": "Anchor_Financial",
        "Sector": "Financials",
    },
    "MS": {
        "Name": "Morgan Stanley",
        "Category": "Anchor_Financial",
        "Sector": "Financials",
    },
    # --- 2020 COVID Crisis (Distressed Specialty Financials & Energy) ---
    "MFA": {
        "Name": "MFA Financial",
        "Category": "Distressed_2020",
        "Sector": "Real_Estate_Finance",
    },
    "IVR": {
        "Name": "Invesco Mortgage Capital",
        "Category": "Distressed_2020",
        "Sector": "Real_Estate_Finance",
    },
    "TWO": {
        "Name": "Two Harbors Investment",
        "Category": "Distressed_2020",
        "Sector": "Real_Estate_Finance",
    },
    "HTZ": {
        "Name": "Hertz Global Holdings",
        "Category": "Distressed_2020",
        "Sector": "Consumer_Services",
    },
    "CHK": {
        "Name": "Chesapeake Energy",
        "Category": "Distressed_2020",
        "Sector": "Energy",
    },
    # --- 2023 Regional Banking Crisis (Distressed & Regional Anchors) ---
    "SIVB": {
        "Name": "Silicon Valley Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "FRCB": {
        "Name": "First Republic Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "SBNY": {
        "Name": "Signature Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "PACW": {
        "Name": "PacWest Bancorp",
        "Category": "Distressed_2023",
        "Sector": "Financials",
    },
    "WAL": {
        "Name": "Western Alliance",
        "Category": "Regional_Bank",
        "Sector": "Financials",
    },
    "KEY": {"Name": "KeyCorp", "Category": "Regional_Bank", "Sector": "Financials"},
    "FITB": {
        "Name": "Fifth Third Bancorp",
        "Category": "Regional_Bank",
        "Sector": "Financials",
    },
    # --- Market Indices & Sector Benchmarks ---
    "^GSPC": {
        "Name": "S&P 500 Index",
        "Category": "Benchmark",
        "Sector": "Broad_Market",
    },
    "^VIX": {
        "Name": "CBOE Volatility Index",
        "Category": "Benchmark",
        "Sector": "Volatility",
    },
    "XLF": {
        "Name": "Financial Select Sector SPDR",
        "Category": "Benchmark",
        "Sector": "Financials",
    },
    "XLK": {
        "Name": "Technology Select Sector SPDR",
        "Category": "Control_Sector",
        "Sector": "Technology",
    },
    "XLE": {
        "Name": "Energy Select Sector SPDR",
        "Category": "Control_Sector",
        "Sector": "Energy",
    },
}

TICKERS = list(TICKER_METADATA.keys())
START_DATE = "2006-01-01"  # Fetching extra buffer data prior to GFC 2007
END_DATE = "2023-12-31"

print(
    f"Downloading historical data for {len(TICKERS)} tickers from {START_DATE} to {END_DATE}..."
)

# ------------------------------------------------------------------------------
# 2. BATCH DOWNLOAD HISTORICAL PRICES VIA YFINANCE
# ------------------------------------------------------------------------------
# Use yf.download for fast, vectorized batch retrieval
raw_data = yf.download(
    tickers=TICKERS,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    group_by="ticker",
    auto_adjust=False,
    threads=True,
)

# ------------------------------------------------------------------------------
# 3. TRANSFORM & CLEAN INTO TIDY LONG-FORMAT DATAFRAME
# ------------------------------------------------------------------------------
records = []
missing_tickers = []

for ticker in TICKERS:
    try:
        # Check if ticker returned data
        if ticker in raw_data.columns.levels[0]:
            df_ticker = raw_data[ticker].copy().dropna(how="all")

            if df_ticker.empty:
                missing_tickers.append(ticker)
                continue

            df_ticker.reset_index(inplace=True)
            df_ticker["Ticker"] = ticker
            df_ticker["Name"] = TICKER_METADATA[ticker]["Name"]
            df_ticker["Category"] = TICKER_METADATA[ticker]["Category"]
            df_ticker["Sector"] = TICKER_METADATA[ticker]["Sector"]

            # Calculate daily percent log returns on Adjusted Close
            df_ticker["Adj Close"] = df_ticker["Adj Close"].astype(float)
            df_ticker["Log_Return"] = np.log(
                df_ticker["Adj Close"] / df_ticker["Adj Close"].shift(1)
            )

            records.append(df_ticker)
        else:
            missing_tickers.append(ticker)
    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        missing_tickers.append(ticker)

# Concatenate all ticker data into a single master long-format DataFrame
master_df = pd.concat(records, ignore_index=True)

# Format columns
master_df.rename(columns={"Date": "Date", "Adj Close": "Adj_Close"}, inplace=True)
master_df.sort_values(by=["Ticker", "Date"], inplace=True)

# ------------------------------------------------------------------------------
# 4. SUMMARY & VALIDATION
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print(f"SUCCESSFULLY DOWNLOADED DATA FOR {len(master_df['Ticker'].unique())} TICKERS.")
if missing_tickers:
    print(
        f"WARNING: The following tickers returned no data or are delisted on Yahoo: {missing_tickers}"
    )
    print(
        "Note: For delisted entities missing from Yahoo, offline CSV patches will be merged."
    )
print("=" * 60)

# Display sample output
print("\nMaster Dataset Preview:")
print(master_df[["Date", "Ticker", "Sector", "Adj_Close", "Log_Return"]].head())

# ------------------------------------------------------------------------------
# 5. EXPORT TO CSV AND PARQUET
# ------------------------------------------------------------------------------
output_dir = "../data/raw"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "market_data_raw.csv")
parquet_path = os.path.join(output_dir, "market_data_raw.parquet")

# Save as CSV
master_df.to_csv(csv_path, index=False)
print(
    f"\n[Saved] Raw CSV created at: {csv_path} ({os.path.getsize(csv_path) / 1e6:.2f} MB)"
)

# Save as Parquet (Requires 'pyarrow' or 'fastparquet')
try:
    master_df.to_parquet(parquet_path, index=False)
    print(
        f"[Saved] Raw Parquet created at: {parquet_path} ({os.path.getsize(parquet_path) / 1e6:.2f} MB)"
    )
except ImportError:
    print(
        "\nTip: Install 'pyarrow' using `pip install pyarrow` to save directly in Parquet format."
    )

$BSC: possibly delisted; no price data found  (1d 2006-01-01 -> 2023-12-31)
[                       0%                       ]

[****                   8%                       ]  2 of 25 completed$CHK: possibly delisted; no timezone found
[********************* 44%                       ]  11 of 25 completed$LEHMQ: possibly delisted; no price data found  (1d 2006-01-01 -> 2023-12-31)
[**********************52%                       ]  13 of 25 completed$SBNY: possibly delisted; no price data found  (1d 2006-01-01 -> 2023-12-31) (Yahoo error = "Data doesn't exist for startDate = 1136091600, endDate = 1703998800")
[**********************72%**********             ]  18 of 25 completed$PACW: possibly delisted; no timezone found
[**********************72%**********             ]  18 of 25 completed$SIVB: possibly delisted; no timezone found
[*********************100%***********************]  25 of 25 completed

6 Failed downloads:
['BSC', 'LEHMQ']: possibly delisted; no price data found  (1d 2006-01-01 -> 2023-12-31)
['CHK', 'PACW', 'SIVB']: possibly delisted; no timezone found
['SBNY']: possibly delisted; no price


SUCCESSFULLY DOWNLOADED DATA FOR 19 TICKERS.
Note: For delisted entities missing from Yahoo, offline CSV patches will be merged.

Master Dataset Preview:
Price       Date Ticker      Sector   Adj_Close  Log_Return
0     2006-01-03    AIG  Financials  829.507202         NaN
1     2006-01-04    AIG  Financials  830.698914    0.001436
2     2006-01-05    AIG  Financials  831.771118    0.001290
3     2006-01-06    AIG  Financials  835.345459    0.004288
4     2006-01-09    AIG  Financials  831.413818   -0.004718

[Saved] Raw CSV created at: ../data/raw/market_data_raw.csv (14.30 MB)
[Saved] Raw Parquet created at: ../data/raw/market_data_raw.parquet (3.18 MB)


In [ ]:
# NO LONGER NEEDED: The following code snippet is now redundant since the CSV and Parquet files are already saved in the previous steps. It was originally intended to read the CSV and save it as Parquet, but this is now handled directly in the export step above.

import os

import pandas as pd

# Define exact paths
csv_path = "./data/raw/market_data_raw.csv"
parquet_path = "./data/raw/market_data_raw.parquet"

# Fallback check if CSV is in ./data/ instead of ./data/raw/
if not os.path.exists(csv_path) and os.path.exists("./data/market_data_raw.csv"):
    csv_path = "./data/market_data_raw.csv"

print(f"Reading CSV from: {csv_path}")

# Load the CSV
df = pd.read_csv(csv_path)
df["Date"] = pd.to_datetime(df["Date"])

# Ensure target folder exists
os.makedirs("./data/raw", exist_ok=True)

# Write directly to ./data/raw/
df.to_parquet(parquet_path, engine="pyarrow", index=False)

# Verify
if os.path.exists(parquet_path):
    size_mb = os.path.getsize(parquet_path) / 1e6
    print(
        f"SUCCESS! Parquet file written to: {parquet_path} (Size: {size_mb:.2f}" " MB)"
    )
    print("\nContents of ./data/raw/:")
    print(os.listdir("./data/raw/"))
else:
    print("Failed to write Parquet file.")